**Perform data processing, feature scaling, missing value handling and feature selection on the Titanic dataset using Python Libraries.**

In [1]:
import pandas as pd
url="https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df=pd.read_csv(url)
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


## Data Processing: Missing Value Handling

In [2]:
print('Missing values before handling:')
print(df.isnull().sum())

Missing values before handling:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


### Handling Missing Values
- **Age**: Impute with the median, as age can be skewed.
- **Embarked**: Impute with the mode, as it's a categorical feature.
- **Cabin**: Drop this column due to a high number of missing values.

In [3]:
# Impute 'Age' with the median, if 'Age' column exists
if 'Age' in df.columns:
    df['Age'] = df['Age'].fillna(df['Age'].median())

# 'Embarked' column has already been handled by feature engineering, so this imputation is removed.

# Drop 'Cabin' column, if 'Cabin' column exists
if 'Cabin' in df.columns:
    df.drop('Cabin', axis=1, inplace=True)

print('\nMissing values after handling:')
print(df.isnull().sum())


Missing values after handling:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       2
dtype: int64


## Feature Engineering

In [19]:
# Convert 'Sex' to numerical representation (0 for female, 1 for male)
# Only if 'Sex' column exists and is of object type (contains strings like 'male', 'female')
if 'Sex' in df.columns:
    df['Sex'] = df['Sex'].map({'female': 0, 'male': 1})

# One-hot encode 'Embarked' column
# Only if 'Embarked' column exists
if 'Embarked' in df.columns:
    df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

# Drop irrelevant columns, only if they exist
columns_to_drop = ['PassengerId', 'Name', 'Ticket']
for col in columns_to_drop:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

In [20]:
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,1,22.0,1,0,7.2500,False,True
1,1,1,0,38.0,1,0,71.2833,False,False
2,1,3,0,26.0,0,0,7.9250,False,True


## Feature Scaling

In [5]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Identify numerical features for scaling
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical features
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print('\nDataFrame after feature scaling:')
print(X.head())


DataFrame after feature scaling:
     Pclass     Sex       Age     SibSp     Parch      Fare  Embarked_Q  \
0  0.827377    male -0.565736  0.432793 -0.473674 -0.502445       False   
1 -1.566107  female  0.663861  0.432793 -0.473674  0.786845       False   
2  0.827377  female -0.258337 -0.474545 -0.473674 -0.488854       False   
3 -1.566107  female  0.433312  0.432793 -0.473674  0.420730       False   
4  0.827377    male  0.433312 -0.474545 -0.473674 -0.486337       False   

   Embarked_S  
0        True  
1       False  
2        True  
3        True  
4        True  


## Feature Selection (Example - using correlation with target)

In [21]:
# Calculate correlations only for numeric columns
correlations = df_scaled.corr(numeric_only=True)['Survived'].abs().sort_values(ascending=False)
print('\nFeature correlations with Survived (absolute value):')
print(correlations)


Feature correlations with Survived (absolute value):
Survived      1.000000
Pclass        0.338481
Fare          0.257307
Embarked_S    0.155660
Parch         0.081629
Age           0.064910
SibSp         0.035322
Embarked_Q    0.003650
Name: Survived, dtype: float64
